In [2]:
import pandas as pd
import numpy as np
import torch

In [3]:
class Feedforward(torch.nn.Module):
        def __init__(self, input_size, hidden_size, dropout):
            super(Feedforward, self).__init__()
            self.input_size = input_size
            self.hidden_size  = hidden_size
            self.fc1 = torch.nn.Linear(self.input_size, self.hidden_size)
            self.fc2 = torch.nn.Linear(self.hidden_size, self.hidden_size)
            self.fc3 = torch.nn.Linear(self.hidden_size, self.hidden_size)
            self.fc4 = torch.nn.Linear(self.hidden_size, self.hidden_size)
            self.fc5 = torch.nn.Linear(self.hidden_size, self.hidden_size)
            #self.relu = torch.nn.ReLU()
            self.relu = torch.nn.LeakyReLU(0.1)
            self.fc_out = torch.nn.Linear(self.hidden_size, 1)
            self.sigmoid = torch.nn.Sigmoid()
            self.dropout = torch.nn.Dropout(dropout)
            
        def forward(self, x):
            x = self.fc1(x)
            x = self.dropout(x)
            x = self.relu(x)
            x = self.fc2(x)
            x = self.dropout(x)
            x = self.relu(x)
            x = self.fc3(x)
            x = self.dropout(x)
            x = self.relu(x)
            x = self.fc4(x)
            x = self.dropout(x)
            x = self.relu(x)
            x = self.fc5(x)
            x = self.dropout(x)
            x = self.relu(x)
            output = self.fc_out(x)
            output = self.sigmoid(output)
            return output

In [4]:
data = pd.read_csv('/Users/nkerstingadxnet.com/Documents/Higgs/orig/atlas-higgs-challenge-2014-v2.csv')
data['Binary_Label'] = data['Label'].map({'s':1,'b':0})

In [5]:
nonphi_columns = [c for c in data.columns if "phi" in c]
for col in nonphi_columns:
    data.drop(col, axis=1, inplace=True)
data.head()

,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_all_pt,Weight,Label,KaggleSet,KaggleWeight,Binary_Label
0,100000,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,...,67.435,2.150,46.062,1.24,113.497,0.000814,s,t,0.002653,1
1,100001,160.937,68.768,103.235,48.146,-999.00,-999.000,-999.000,3.473,2.078,...,46.226,0.725,-999.000,-999.00,46.226,0.681042,b,t,2.233584,0
2,100002,-999.000,162.172,125.953,35.635,-999.00,-999.000,-999.000,3.148,9.336,...,44.251,2.053,-999.000,-999.00,44.251,0.715742,b,t,2.347389,0
3,100003,143.905,81.417,80.943,0.414,-999.00,-999.000,-999.000,3.310,0.414,...,-999.000,-999.000,-999.000,-999.00,-0.000,1.660654,b,t,5.446378,0
4,100004,175.864,16.915,134.805,16.405,-999.00,-999.000,-999.000,3.891,16.405,...,-999.000,-999.000,-999.000,-999.00,0.000,1.904263,b,t,6.245333,0


In [6]:
# now replace all the '-999' values with NULL so we can properly handle them
#nulled_data = data.replace(-999.0, np.nan)
nulled_data = data.replace(-999.0, 0)
train_data = nulled_data.loc[nulled_data['KaggleSet'] == 't'].copy(deep=True)
public_test_data = nulled_data.loc[nulled_data['KaggleSet'] == 'b'].copy(deep=True)
private_test_data = nulled_data.loc[nulled_data['KaggleSet'] == 'v'].copy(deep=True)

In [7]:
orig_train_data = train_data.copy(deep=True)
orig_public_test_data = public_test_data.copy(deep=True)
orig_private_test_data = private_test_data.copy(deep=True)


# now replace the null values with column averages
train_data.drop('EventId', axis=1, inplace=True)
train_data.drop('Label', axis=1, inplace=True)
train_data.drop('Binary_Label', axis=1, inplace=True)
train_data.drop('Weight', axis=1, inplace=True)
train_data.drop('KaggleWeight', axis=1, inplace=True)
train_data.drop('KaggleSet', axis=1, inplace=True)
for col in train_data:
    train_data[col].fillna(train_data[col].mean(), inplace=True)
#train_data.isnull().sum()


public_test_data.drop('EventId', axis=1, inplace=True)
public_test_data.drop('Label', axis=1, inplace=True)
public_test_data.drop('Binary_Label', axis=1, inplace=True)
public_test_data.drop('Weight', axis=1, inplace=True)
public_test_data.drop('KaggleWeight', axis=1, inplace=True)
public_test_data.drop('KaggleSet', axis=1, inplace=True)
for col in public_test_data:
    public_test_data[col].fillna(public_test_data[col].mean(), inplace=True)
#public_test_data.isnull().sum()


private_test_data.drop('EventId', axis=1, inplace=True)
private_test_data.drop('Label', axis=1, inplace=True)
private_test_data.drop('Binary_Label', axis=1, inplace=True)
private_test_data.drop('Weight', axis=1, inplace=True)
private_test_data.drop('KaggleWeight', axis=1, inplace=True)
private_test_data.drop('KaggleSet', axis=1, inplace=True)
for col in private_test_data:
    private_test_data[col].fillna(private_test_data[col].mean(), inplace=True)
#private_test_data.isnull().sum()

In [8]:
# now let's normalize
normed_train_data = (train_data - train_data.min())/(train_data.max() - train_data.min())

normed_public_test_data = (public_test_data - public_test_data.min())/(public_test_data.max() - public_test_data.min())

normed_private_test_data = (private_test_data - private_test_data.min())/(private_test_data.max() - private_test_data.min())


In [59]:
normed_train_data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 250000 entries, 0 to 249999
Data columns (total 24 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   DER_mass_MMC                 250000 non-null  float64
 1   DER_mass_transverse_met_lep  250000 non-null  float64
 2   DER_mass_vis                 250000 non-null  float64
 3   DER_pt_h                     250000 non-null  float64
 4   DER_deltaeta_jet_jet         250000 non-null  float64
 5   DER_mass_jet_jet             250000 non-null  float64
 6   DER_prodeta_jet_jet          250000 non-null  float64
 7   DER_deltar_tau_lep           250000 non-null  float64
 8   DER_pt_tot                   250000 non-null  float64
 9   DER_sum_pt                   250000 non-null  float64
 10  DER_pt_ratio_lep_tau         250000 non-null  float64
 11  DER_lep_eta_centrality       250000 non-null  float64
 12  PRI_tau_pt                   250000 non-null  float64
 13 

In [9]:
train_input_data = []
for i in range(len(orig_train_data)):
   train_input_data.append([torch.tensor(normed_train_data.iloc[i], dtype=torch.float), torch.tensor(orig_train_data['Binary_Label'].iloc[i] , dtype=torch.float)])
valid_input_data = []
for i in range(len(orig_public_test_data)):
   valid_input_data.append([torch.tensor(normed_public_test_data.iloc[i], dtype=torch.float), torch.tensor(orig_public_test_data['Binary_Label'].iloc[i] , dtype=torch.float)])

In [10]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_input_data, batch_size=128, shuffle=True)
valid_dataloader = DataLoader(valid_input_data, batch_size=128, shuffle=True)

In [11]:
model = Feedforward(24, 600, 0.05)
criterion = torch.nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.01)

In [12]:
import time
import copy
from datetime import datetime

now = datetime.now()
dt_string = now.strftime("%d%m%Y%H%M%S")
logfile = open(dt_string + ".log",'w')

start_time = time.time()
epoch = 20
maxvalcount = 5
minval_loss = np.inf
valcount = maxvalcount
best_model = model
for epoch in range(epoch):
    print("EPOCH: ", epoch)
    model.train()
    for i,batch in enumerate(train_dataloader):
        inputs, output = batch
        optimizer.zero_grad()
        # Forward pass
        y_pred = model(inputs)
        # Compute Loss
        loss = criterion(y_pred.squeeze(), output)
        if i % 100 == 0:
            logstring = 'Batch {}: train loss: {}'.format(i, loss.item())
            print(logstring)
            logfile.write(logstring + '\n')
        # Backward pass
        loss.backward()
        optimizer.step()
    # compute validation loss
    model.eval()
    with torch.set_grad_enabled(False):
        val_loss = 0
        for i,batch in enumerate(valid_dataloader):
            inputs, output = batch
            y_pred = model(inputs)
            y_pred = model(inputs)
            loss = criterion(y_pred.squeeze(), output)
            val_loss += loss
            if i % 100 == 0:
                logstring = 'Validation Batch {}: loss: {}'.format(i, loss.item())
                print(logstring)
                logfile.write(logstring + '\n')
        avg_val_loss = val_loss / len(valid_dataloader)
        valstring = f"AVERAGE BATCH VAL LOSS = {avg_val_loss}"
        print(valstring)
        logfile.write(valstring + '\n')
    if avg_val_loss < minval_loss:
        minval_loss = avg_val_loss
        best_model = copy.deepcopy(model)
        valcount = maxvalcount
    else:
        valcount -= 1
    if valcount == 0:
        endmsg = f"Validation Loss failed to decrease in {maxvalcount} epochs, exiting with best model, val loss = {minval_loss}"
        print(endmsg)
        logfile.write(endmsg + '\n')
        break

end_time = time.time()
timestring = f"Time elapsed in training: {end_time - start_time} seconds"
print(timestring)
logfile.write(timestring + '\n')

EPOCH:  0
Batch 0: train loss: 0.6939132809638977
Batch 100: train loss: 0.6632571816444397
Batch 200: train loss: 0.6574290990829468
Batch 300: train loss: 0.6586119532585144
Batch 400: train loss: 0.6301531791687012
Batch 500: train loss: 0.6532390117645264
Batch 600: train loss: 0.6441534757614136
Batch 700: train loss: 0.6342890858650208
Batch 800: train loss: 0.619281530380249
Batch 900: train loss: 0.658097505569458
Batch 1000: train loss: 0.6330832242965698
Batch 1100: train loss: 0.6532653570175171
Batch 1200: train loss: 0.6583679914474487
Batch 1300: train loss: 0.6275248527526855
Batch 1400: train loss: 0.6527047753334045
Batch 1500: train loss: 0.668343186378479
Batch 1600: train loss: 0.6531516909599304
Batch 1700: train loss: 0.6380125284194946
Batch 1800: train loss: 0.6936875581741333
Batch 1900: train loss: 0.6378073692321777
Validation Batch 0: loss: 0.6578571200370789
Validation Batch 100: loss: 0.6575490236282349
Validation Batch 200: loss: 0.6531433463096619
Valida

Batch 1300: train loss: 0.49201956391334534
Batch 1400: train loss: 0.43688681721687317
Batch 1500: train loss: 0.442095547914505
Batch 1600: train loss: 0.5520511269569397
Batch 1700: train loss: 0.4940251410007477
Batch 1800: train loss: 0.510895848274231
Batch 1900: train loss: 0.46572715044021606
Validation Batch 0: loss: 0.49045148491859436
Validation Batch 100: loss: 0.6170564293861389
Validation Batch 200: loss: 0.509824812412262
Validation Batch 300: loss: 0.5237995982170105
Validation Batch 400: loss: 0.48953890800476074
Validation Batch 500: loss: 0.45610296726226807
Validation Batch 600: loss: 0.48877838253974915
Validation Batch 700: loss: 0.6119838356971741
AVERAGE BATCH VAL LOSS = 0.5265254974365234
EPOCH:  7
Batch 0: train loss: 0.48958754539489746
Batch 100: train loss: 0.4965895414352417
Batch 200: train loss: 0.4476437568664551
Batch 300: train loss: 0.5410751700401306
Batch 400: train loss: 0.425415962934494
Batch 500: train loss: 0.4822673797607422
Batch 600: train 

Validation Batch 500: loss: 0.4932243227958679
Validation Batch 600: loss: 0.5452737212181091
Validation Batch 700: loss: 0.514356255531311
AVERAGE BATCH VAL LOSS = 0.555183470249176
EPOCH:  13
Batch 0: train loss: 0.36356866359710693
Batch 100: train loss: 0.4840329587459564
Batch 200: train loss: 0.3349227011203766
Batch 300: train loss: 0.4521879255771637
Batch 400: train loss: 0.4488215744495392
Batch 500: train loss: 0.3475363254547119
Batch 600: train loss: 0.48818737268447876
Batch 700: train loss: 0.3836144208908081
Batch 800: train loss: 0.4435311555862427
Batch 900: train loss: 0.42256781458854675
Batch 1000: train loss: 0.40732720494270325
Batch 1100: train loss: 0.4263036847114563
Batch 1200: train loss: 0.39793089032173157
Batch 1300: train loss: 0.4901430904865265
Batch 1400: train loss: 0.4009093642234802
Batch 1500: train loss: 0.38082998991012573
Batch 1600: train loss: 0.5053346753120422
Batch 1700: train loss: 0.4362855553627014
Batch 1800: train loss: 0.367319256067

Batch 700: train loss: 0.3772420287132263
Batch 800: train loss: 0.3888138234615326
Batch 900: train loss: 0.35250434279441833
Batch 1000: train loss: 0.3030570447444916
Batch 1100: train loss: 0.417118102312088
Batch 1200: train loss: 0.3328879177570343
Batch 1300: train loss: 0.39253440499305725
Batch 1400: train loss: 0.44840794801712036
Batch 1500: train loss: 0.39656755328178406
Batch 1600: train loss: 0.41722556948661804
Batch 1700: train loss: 0.3950056731700897
Batch 1800: train loss: 0.395925372838974
Batch 1900: train loss: 0.3629803955554962
Validation Batch 0: loss: 0.4189511239528656
Validation Batch 100: loss: 0.33250489830970764
Validation Batch 200: loss: 0.3570936620235443
Validation Batch 300: loss: 0.3662481904029846
Validation Batch 400: loss: 0.36324185132980347
Validation Batch 500: loss: 0.41968968510627747
Validation Batch 600: loss: 0.3776662051677704
Validation Batch 700: loss: 0.4533129334449768
AVERAGE BATCH VAL LOSS = 0.4067927896976471
Time elapsed in trai

In [22]:
torch.save(best_model, '/Users/nkerstingadxnet.com/Documents/Higgs/output/mlp.600.nophi.pt')

In [23]:
import math
def AMS(s, b):
    """ Approximate Median Significance defined as:
        AMS = sqrt(
                2 { (s + b + b_r) log[1 + (s/(b+b_r))] - s}
              )        
    where b_r = 10, b = background, s = signal, log is natural logarithm """
    
    br = 10.0
    radicand = 2 *( (s+b+br) * math.log (1.0 + s/(b+br)) -s)
    if radicand < 0:
        print('radicand is negative. Exiting')
        exit()
    else:
        return math.sqrt(radicand)

In [24]:
# Training performance
best_model.eval()
y_pred = best_model(torch.tensor(normed_train_data.values, dtype=torch.float)).tolist()
y_gold = orig_train_data['Label']
y_weight = orig_train_data['KaggleWeight']
s = 0
b = 0
tp = 0
fp = 0
for i,y in enumerate(y_pred):
    if y[0] > 0.5:
        if y_gold.iloc[i] == 's':
            s += y_weight.iloc[i]
            tp += 1
        else:
            b += y_weight.iloc[i]
            fp += 1
outstring = f"TRAINING: (S,B) = ({s:.3f},{b:.3f}), AMS={AMS(s,b):.3f}, Unweighted Precision = {tp/(tp+fp):.3f}"
print(outstring)
logfile.write(outstring + '\n')

TRAINING: (S,B) = (260.255,7055.661), AMS=3.077, Unweighted Precision = 0.889


ValueError: I/O operation on closed file.

In [25]:
# Validation performance
y_pred = best_model(torch.tensor(normed_public_test_data.values, dtype=torch.float)).tolist()
y_gold = orig_public_test_data['Label']
y_weight = orig_public_test_data['KaggleWeight']
s = 0
b = 0
tp = 0
fp = 0
for i,y in enumerate(y_pred):
    if y[0] > 0.5:
        if y_gold.iloc[i] == 's':
            s += y_weight.iloc[i]
            tp += 1
        else:
            b += y_weight.iloc[i]
            fp += 1
outstring = f"VALID: (S,B) = ({s:.3f},{b:.3f}), AMS={AMS(s,b):.3f}, Unweighted Precision = {tp/(tp+fp):.3f}"
print(outstring)
logfile.write(outstring + '\n')

VALID: (S,B) = (427.961,23656.495), AMS=2.774, Unweighted Precision = 0.754


ValueError: I/O operation on closed file.

In [26]:
# Test set performance
y_pred = best_model(torch.tensor(normed_private_test_data.values, dtype=torch.float)).tolist()
y_gold = orig_private_test_data['Label']
y_weight = orig_private_test_data['KaggleWeight']
s = 0
b = 0
tp = 0
fp = 0
for i,y in enumerate(y_pred):
    if y[0] > 0.5:
        if y_gold.iloc[i] == 's':
            s += y_weight.iloc[i]
            tp += 1
        else:
            b += y_weight.iloc[i]
            fp += 1
outstring = f"TEST: (S,B) = ({s:.3f},{b:.3f}), AMS={AMS(s,b):.3f}, Unweighted Precision = {tp/(tp+fp):.3f}"
print(outstring)
logfile.write(outstring + '\n')

TEST: (S,B) = (268.659,16633.524), AMS=2.077, Unweighted Precision = 0.754


ValueError: I/O operation on closed file.

In [21]:
logfile.close()

In [27]:
np.inf

inf

In [28]:
1 < np.inf

True